# FHRPY — CTG viewer & analysis demo

Open fetal heart-rate recordings, show the **real WMFB baseline**,
**accel/deceleration** & **contraction** zones and **false-signal** detection —
or display the **expert labels** instead — all inline under **VSCode** and
**Google Colab**.

> **Run the setup cell below first.** It makes `fhrpy` importable on its own —
> no manual `pip install` from a checkout of
> [data-coeur/FHRPY](https://github.com/data-coeur/FHRPY).

In [ ]:
# === Setup — run this cell first (makes the notebook self-contained) ===
# Ensures `fhrpy` is importable both from a checkout (VSCode) and standalone (Colab).
try:
    import fhrpy
except ModuleNotFoundError:
    import sys, pathlib
    # 1) Inside a checkout of the repo: add the repo root (which holds the
    #    `fhrpy/` package) to sys.path. No install needed; deps are already present.
    root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (p / "fhrpy" / "__init__.py").exists()), None)
    if root:
        sys.path.insert(0, str(root))
    else:
        # 2) Standalone (e.g. Google Colab): install the released package from
        #    GitHub. The `main` branch is public, so NO token is required.
        %pip install -q "fhrpy[viewer] @ git+https://github.com/data-coeur/FHRPY.git@main"
    import fhrpy

print("fhrpy", fhrpy.__version__, "ready")

## 1. Bundled FHRMA examples

A small, self-contained subset of the public **FHRMA** datasets ships with FHRPY
(all with their start timestamp zeroed, so they open at `00h00`). Each example
carries its **expert labels** in a companion `.fhrh` marker file.

In [ ]:
import fhrpy.datasets as ds

for e in ds.list_examples():
    print(f"{e['name']:15s} {e['category']:9s} {e['duration_min']:6.1f} min  -  {e['description']}")

## 2. Open an example showing the **expert labels**

`source="expert"` displays the experts' annotations directly:
* morphology examples → expert **baseline** + **acceleration** (green) /
  **deceleration** (red) zones;
* false-signal examples → expert **false-signal** zones (light pink) + the
  **expulsion** instant.

Just change `EXAMPLE` to any name printed above to explore the others.

In [ ]:
from fhrpy.viewer import FHRViewer, link_scroll

EXAMPLE = "morpho_train21"          # <- change me (see the list in section 1)
ds.load_example(EXAMPLE, source="expert", height=460)

## 3. False signals — expert labels + the protected `£Expulsion` mark

For these examples the grey/pink zones are the **expert** false-signal episodes,
and the blue **Expulsion** line is a *protected* marker (its `£` prefix means it
cannot be edited or deleted in the viewer), so you never have to pass the
expulsion time as a parameter.

> **False-signal detection requirements.** Doppler false-signal detection expects
> `FHR1` = the **Doppler** channel and `FHR2` = the **scalp** (direct fetal ECG)
> channel. The maternal-heart-rate channel (`MHR`) must be **time-aligned** to the
> FHR before detection: on Philips monitors the maternal pulse lags by about
> **12.5 s** when measured with the SpO2 **oximeter**, or about **5 s** when derived
> from the **TOCO/belt** — shift `MHR` by that delay first.

In [ ]:
ds.load_example("fs_dopmhr_01", source="expert", channels=["FHR1", "MHR"], height=460)

## 4. Compute the methods **head-less** (no viewer), then use the results

You can run the analysis without rendering anything, inspect the numbers, and
*then* feed them to a viewer / compare with the labels / export.

In [ ]:
from fhrpy.io import read_fhr
from fhrpy.baseline import analyze
from fhrpy.falsesignal import detect_false_signals

# WMFB baseline + morphology, no UI:
rec = read_fhr(ds.example_path("morpho_train21"))
ma = analyze(rec)
print("baseline pts:", len(ma["baseline"]),
      "| acc:", len(ma["accelerations"]), "| dec:", len(ma["decelerations"]))

# False-signal detection, no UI (FHR1 = Doppler, MHR aligned — see section 3):
fsrec = read_fhr(ds.example_path("fs_dopmhr_01"))
fs = detect_false_signals(fsrec, kind="doppler")
print("false-signal episodes:", len(fs["segments"]),
      "| P(false) max:", round(float(fs["prob"].max()), 3))

## 5. Compare **expert labels vs the method**, scroll-synchronised

Display the same record twice — once with the expert labels, once with FHRPY's
prediction — and `link_scroll(...)` locks both time windows to the **same
instant**: scroll, page or wheel-scroll either one and the other follows. That
makes the differences between *label* and *prediction* easy to spot.

In [ ]:
from IPython.display import display, HTML

label = ds.load_example("morpho_train21", source="expert", height=340)
pred  = ds.load_example("morpho_train21", source="method", height=340)

display(HTML("<b>Expert labels</b>"));            display(label)
display(HTML("<b>FHRPY prediction (WMFB)</b>"));  display(pred)

link_scroll(label, pred)   # scroll one -> the other follows to the same time

## 6. Drive the viewer dynamically from Python

The viewer is a **controllable object**. Display it, then run the next cell to
steer it live and register event listeners.

In [ ]:
v = ds.load_example("morpho_train21", source="expert", height=440)
v

In [ ]:
v.set_scale(3)                  # 3 cm/min paper speed
v.set_safe_zone(110, 150)       # move the central "safe" band
v.scroll_to(0.5)                # jump to the middle of the record
v.set_channel_visible("MHR", False)
v.on("scroll", lambda d: print("scrolled to", round(d.get("time", 0) / 60, 1), "min"))
# other controls: set_range(min,max), set_zones_visible, set_contractions_visible,
# set_false_signals_visible, set_interpolate, set_markers, get markers, print()

## 7. Options + export a standalone HTML with the **same parameters**

Every display option can be set at construction (top-grid bounds, safe band,
channels, paper speed, gap interpolation, height …) and exported to a single
offline HTML page — no server, no PHP. Use the printer button in the toolbar to
get a multi-page **A4-landscape PDF** (≈ 1 cm/min, 20 bpm/cm).

In [ ]:
v = ds.load_example(
    "fs_dopmhr_01", source="expert",
    channels=["FHR1", "MHR"], scale=1, height=460,
    rcf_min=50, rcf_max=210,      # configurable top-grid bounds
    safe_min=110, safe_max=160,   # configurable central safe band
    interpolate=True,
)
v.to_html("fhrma_example.html")   # offline page with exactly these parameters
print("wrote fhrma_example.html")
v